In [1]:
import os
import gc
import random
import timm
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, log_loss
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from scipy.stats import randint, uniform
from sklearn.decomposition import PCA


import xgboost as xgb
import lightgbm as lgb
import catboost as cb

#  Settings 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
N = 10   # ways
K = 3    # shots
Q = 4    # queries per class
EPISODES = 1000  # Few-shot episodes
PCA_DIM = 1280

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#  Dataset 
class ProtoDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = np.array(file_paths)
        self.labels = np.array(labels)
        self.transform = transform
        self.classes = np.unique(labels)
        self.class_to_indices = {c: np.where(labels == c)[0] for c in self.classes}

    def sample_episode(self, N, K, Q):
        selected_classes = np.random.choice(self.classes, N, replace=False)
        support_images, support_labels, query_images, query_labels = [], [], [], []

        for i, cls in enumerate(selected_classes):
            idxs = np.random.choice(self.class_to_indices[cls], K + Q, replace=False)
            support_idxs, query_idxs = idxs[:K], idxs[K:]
            for idx in support_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                support_images.append(img)
                support_labels.append(i)
            for idx in query_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                query_images.append(img)
                query_labels.append(i)
        return (
            torch.stack(support_images),
            torch.tensor(support_labels),
            torch.stack(query_images),
            torch.tensor(query_labels)
        )

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

In [3]:
#  Load Dataset 
DATA_PATH = "C:/Users/MSC055/Desktop/Sabrina/CSE711/Eye Image Dataset"
CLASSES = sorted(os.listdir(DATA_PATH))

file_paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls)
    for f in os.listdir(cls_path):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            file_paths.append(os.path.join(cls_path, f))
            labels.append(idx)

file_paths = np.array(file_paths)
labels = np.array(labels)

X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

print(f"Train_size: {len(X_train)} | Val_size: {len(X_val)} | Test_size: {len(X_test)}")


Train_size: 3734 | Val_size: 800 | Test_size: 801


In [4]:
#  Prototypical Network 
class ProtoNetwork(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,      # removing classifier head
            global_pool="avg"
        )

    def forward(self, x):
        return self.backbone(x)

class ProtoLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, support, support_labels, query, query_labels, model):
        support_emb = model(support)
        query_emb = model(query)
        prototypes = []
        for cls in torch.unique(support_labels):
            prototypes.append(support_emb[support_labels==cls].mean(dim=0))
        prototypes = torch.stack(prototypes)
        dists = torch.cdist(query_emb, prototypes)
        logits = -dists
        return self.ce(logits, query_labels)

#  Feature Extraction Function 
def extract_embeddings(model, file_paths, transform, batch_size=16):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(file_paths), batch_size), desc="Extracting Features"):
            batch_paths = file_paths[i:i+batch_size]
            imgs = []
            for path in batch_paths:
                img = Image.open(path).convert("RGB")
                imgs.append(transform(img))
            imgs = torch.stack(imgs).to(DEVICE)
            feats = model(imgs).cpu().numpy().astype(np.float32)
            embeddings.append(feats)
            del imgs, feats
            gc.collect()
    return np.vstack(embeddings)

In [5]:
#  ViT Backbones for Feature Fusion 
backbones = ["convit_small", "mobilevit_s", "efficientvit_b0"]
# Symbolic Models
symbolic_models = {
    "RandomForest": (
        RandomForestClassifier(random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(5, 50),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", None],
            "bootstrap": [True, False]
        }
    ),
    "XGBoost": (
        xgb.XGBClassifier(eval_metric="mlogloss", random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    
    "LightGBM": (
        lgb.LGBMClassifier(verbose=-1, random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    "CatBoost": (
        cb.CatBoostClassifier(verbose=0,random_state=SEED),
        {
            "iterations": randint(50, 100),
            "depth": randint(3, 12),
            "learning_rate": uniform(0.01, 0.3),
            "l2_leaf_reg": uniform(1, 10),
            "border_count": randint(32, 255),
            "random_strength": uniform(0, 10)
        }
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=SEED),
        {
           "n_estimators": randint(50, 100),
            "learning_rate": uniform(0.01, 1.0)
        }
    )
}


In [ ]:
#  Training + Feature Fusion 

# Prepare ProtoDataset
proto_dataset = ProtoDataset(X_train, y_train, transform=train_transform)

# Train Prototypical Networks for each backbone
proto_models = {}
for name in backbones:
    print(f"\nTraining backbone: {name}")
    proto_model = ProtoNetwork(name).to(DEVICE)
    proto_model.train()
    optimizer = torch.optim.Adam(proto_model.parameters(), lr=1e-4)
    proto_loss = ProtoLoss()

    print("Training Prototypical Network...")
    for episode in range(EPISODES):
        support_x, support_y, query_x, query_y = proto_dataset.sample_episode(N,K,Q)
        support_x, query_x = support_x.to(DEVICE), query_x.to(DEVICE)
        support_y, query_y = support_y.to(DEVICE), query_y.to(DEVICE)
        loss = proto_loss(support_x, support_y, query_x, query_y, proto_model)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if episode % 50 == 0:
            print(f"[{name}] Episode {episode}/{EPISODES} | Prototypical Loss: {loss.item():.4f}")

    proto_models[name] = proto_model
    torch.cuda.empty_cache()
    gc.collect()

#  Feature Fusion + PCA (TRAIN)
def fuse_features_train(models_dict, file_paths, transform, pca_dim=PCA_DIM):
    all_feats = []
    pca_models = {}

    for name, model in models_dict.items():
        print(f"\nExtracting TRAIN features for {name}...")
        feats = extract_embeddings(model, file_paths, transform)

        n_samples, feat_dim = feats.shape
        effective_pca_dim = min(pca_dim, feat_dim, n_samples)

        print(
            f"Applying PCA for {name} | "
            f"samples={n_samples}, feat_dim={feat_dim}, pca_dim={effective_pca_dim}"
        )

        pca = PCA(
            n_components=effective_pca_dim,
            whiten=True,
            random_state=SEED
        )

        feats_pca = pca.fit_transform(feats)

        all_feats.append(feats_pca)
        pca_models[name] = pca

        del feats, feats_pca
        gc.collect()

    fused = np.concatenate(all_feats, axis=1)
    print(f"\nFused TRAIN feature shape: {fused.shape}")

    return fused, pca_models

#  Feature Fusion + PCA (VAL)
def fuse_features_eval(models_dict, file_paths, transform, pca_models):
    all_feats = []

    for name, model in models_dict.items():
        print(f"\nExtracting VAL features for {name}...")
        feats = extract_embeddings(model, file_paths, transform)

        feats_pca = pca_models[name].transform(feats)
        all_feats.append(feats_pca)

        del feats, feats_pca
        gc.collect()

    fused = np.concatenate(all_feats, axis=1)
    print(f"\nFused VAL feature shape: {fused.shape}")

    return fused


# TRAIN FEATURES
X_train_feat, pca_models = fuse_features_train(
    proto_models, X_train, val_transform
)

# VALIDATION FEATURES
X_val_feat = fuse_features_eval(
    proto_models, X_val, val_transform, pca_models
)


#  Train Symbolic Models 
for sym_name, (clf, param_dist) in symbolic_models.items():
    print(f"\nTraining Symbolic Model: {sym_name}")
    search = RandomizedSearchCV(clf, param_distributions=param_dist, n_iter=10, cv=3,
                                scoring="accuracy", random_state=SEED, n_jobs=-1)
    search.fit(X_train_feat, y_train)
    tuned_model = search.best_estimator_
    best_params = search.best_params_

    y_train_pred = tuned_model.predict(X_train_feat)
    y_val_pred = tuned_model.predict(X_val_feat)
    y_train_proba = tuned_model.predict_proba(X_train_feat)
    y_val_proba = tuned_model.predict_proba(X_val_feat)

    train_loss_calc = log_loss(y_train, y_train_proba)
    val_loss_calc = log_loss(y_val, y_val_proba)
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    overall_acc = val_acc
    precision = precision_score(y_val, y_val_pred, average='macro', zero_division=0)
    recall = recall_score(y_val, y_val_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    class_report = classification_report(y_val, y_val_pred, zero_division=0)
    conf_matrix = confusion_matrix(y_val, y_val_pred)

    print(f"\nFeature Fusion + {sym_name} Results:")
    print(f"Train Loss: {train_loss_calc:.4f} | Val Loss: {val_loss_calc:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Accuracy: {overall_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"Best Params: {best_params}")
    print(f"Classification Report:\n{class_report}")
    print(f"Confusion Matrix:\n{conf_matrix}")


Training backbone: convit_small
Training Prototypical Network...
[convit_small] Episode 0/1000 | Prototypical Loss: 1.6569
[convit_small] Episode 50/1000 | Prototypical Loss: 0.8567
[convit_small] Episode 100/1000 | Prototypical Loss: 0.5914
[convit_small] Episode 150/1000 | Prototypical Loss: 0.6625
[convit_small] Episode 200/1000 | Prototypical Loss: 0.6520
[convit_small] Episode 250/1000 | Prototypical Loss: 0.3219
[convit_small] Episode 300/1000 | Prototypical Loss: 0.4622
[convit_small] Episode 350/1000 | Prototypical Loss: 0.3753
[convit_small] Episode 400/1000 | Prototypical Loss: 0.3874
[convit_small] Episode 450/1000 | Prototypical Loss: 0.1555
[convit_small] Episode 500/1000 | Prototypical Loss: 0.4569
[convit_small] Episode 550/1000 | Prototypical Loss: 0.2200
[convit_small] Episode 600/1000 | Prototypical Loss: 0.1276
[convit_small] Episode 650/1000 | Prototypical Loss: 0.1762
[convit_small] Episode 700/1000 | Prototypical Loss: 0.4141
[convit_small] Episode 750/1000 | Pro

Extracting Features: 100%|██████████| 234/234 [01:45<00:00,  2.21it/s]


Applying PCA for convit_small | samples=3734, feat_dim=432, pca_dim=432

Extracting TRAIN features for mobilevit_s...


Extracting Features: 100%|██████████| 234/234 [01:51<00:00,  2.10it/s]


Applying PCA for mobilevit_s | samples=3734, feat_dim=640, pca_dim=640

Extracting TRAIN features for efficientvit_b0...


Extracting Features: 100%|██████████| 234/234 [01:38<00:00,  2.37it/s]


... (5 outputs left out)

Extracting Features: 100%|██████████| 50/50 [00:20<00:00,  2.40it/s]



Fused VAL feature shape: (800, 2352)

Training Symbolic Model: RandomForest

Feature Fusion + RandomForest Results:
Train Loss: 0.3108 | Val Loss: 0.8525 | Train Acc: 0.8856 | Val Acc: 0.7588
Accuracy: 0.7588 | Precision: 0.7639 | Recall: 0.7561 | F1: 0.7559
Best Params: {'bootstrap': True, 'max_depth': 6, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 13, 'n_estimators': 57}
Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.53      0.59        15
           1       0.93      0.92      0.93       226
           2       0.88      0.79      0.83        19
           3       0.66      0.75      0.70       202
           4       0.74      0.63      0.68       153
           5       0.67      0.66      0.66        67
           6       0.61      0.61      0.61        75
           7       0.75      1.00      0.86         3
           8       0.85      0.89      0.87        19
           9       0.89      0.76    

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Feature Fusion + LightGBM Results:
Train Loss: 0.1903 | Val Loss: 0.7248 | Train Acc: 0.9387 | Val Acc: 0.7575
Accuracy: 0.7575 | Precision: 0.8179 | Recall: 0.7614 | F1: 0.7851
Best Params: {'colsample_bytree': np.float64(0.8037724259507192), 'gamma': np.float64(0.8526206184364576), 'learning_rate': np.float64(0.029515477895583855), 'max_depth': 6, 'n_estimators': 74, 'reg_alpha': np.float64(0.9656320330745594), 'reg_lambda': np.float64(0.8083973481164611), 'subsample': np.float64(0.6523068845866853)}
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.67      0.80        15
           1       0.92      0.94      0.93       226
           2       0.88      0.79      0.83        19
           3       0.63      0.72      0.68       202
           4       0.72      0.65      0.68       153
           5       0.73      0.64      0.68        67
           6       0.61      0.60      0.60        75
           7       1.00      1.00  